# Peut-on construire le graphe avec ces fichiers ?

> **Source :** `structure/taxonomy_3.json` (taxonomie),`label/instances.json` (détections), `dataset.csv` (textes).On mesure, on ne corrige pas. Règles qui en découlent : `tuto/04-les-regles.md`.

In [1]:
import json
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

DATA = Path("../data")

topics = json.loads((DATA / "analysis/structure/taxonomy_3.json").read_text())["topics"]
docs = json.loads((DATA / "analysis/label/instances.json").read_text())["documents"]
df = pd.read_csv(DATA / "dataset.csv")

noms = Counter(t["name"] for t in topics)
labels = [lab for d in docs for lab in d["labels"]]

print(f"{len(topics)} topics | {len(docs)} documents | {len(labels)} labels | {len(df)} lignes source")

4840 topics | 100 documents | 9154 labels | 1524 lignes source


## Schéma et jointures

In [2]:
print("topics :", Counter(tuple(sorted(t.keys())) for t in topics))
print("documents :", Counter(tuple(sorted(d.keys())) for d in docs))
print("labels :", Counter(tuple(sorted(lab.keys())) for lab in labels))
print("csv :", df.columns.tolist())

noms_taxo = set(noms)
ids_csv = {str(i) for i in df["id"]}
print(f"\ndocuments -> dataset.csv : {sum(1 for d in docs if str(d['id']) in ids_csv)}/{len(docs)}")
print(f"labels -> taxonomie : {sum(1 for lab in labels if lab['name'] in noms_taxo)}/{len(labels)}")

topics : Counter({('description', 'id', 'level', 'name', 'parent', 'validated'): 4840})
documents : Counter({('id', 'labels'): 100})
labels : Counter({('extract', 'name', 'rationale'): 9154})
csv : ['id', 'content']

documents -> dataset.csv : 100/100
labels -> taxonomie : 9154/9154


Schéma homogène, aucun champ optionnel. Les jointures **entre** fichiers sont parfaites.

## Structure de la hiérarchie

In [3]:
print("niveaux :", dict(Counter(t["level"] for t in topics)))

ids = {t["id"] for t in topics}
parents = [t["parent"] for t in topics if t["parent"]]
print(f"parent par nom : {sum(p in noms_taxo for p in parents)} | par id : {sum(p in ids for p in parents)}")

niveau_de = {t["name"]: t["level"] for t in topics}
print("niveau visé par les labels :", dict(Counter(niveau_de[lab["name"]] for lab in labels)))

niveaux : {0: 4323, 1: 477, 2: 35, 3: 5}
parent par nom : 3591 | par id : 0
niveau visé par les labels : {0: 9154}


Deux points structurants :- `parent` référence un **nom**, jamais un id → la fiabilité des liens dépend de l'unicité des noms.- Les 9154 détections visent **uniquement** le `level 0` → tout nœud parent a 0 instance en propre.

## Nœuds, arêtes, résolution des liens

In [12]:
declares = [t for t in topics if t["parent"]]
resolus = [t for t in declares if noms.get(t["parent"], 0) == 1]
ambigus = [t for t in declares if noms.get(t["parent"], 0) > 1]
casses = [t for t in declares if t["parent"] not in noms]

print(f"nœuds  : {len(topics)}")
print(f"arêtes : {len(declares)} (sans parent : {len(topics) - len(declares)})")
print(f"résolue : {len(resolus)}")
print(f"AMBIGUE : {len(ambigus)} ({round(100 * len(ambigus) / len(declares))}%)")
print(f"cassée : {len(casses)}")

ex = ambigus[0]
print(f'\nex : "{ex["name"]}" -> "{ex["parent"]}" existe {noms[ex["parent"]]} fois')

nœuds  : 4840
arêtes : 3591 (sans parent : 1249)
résolue : 2969
AMBIGUE : 622 (17%)
cassée : 0

ex : "Durcissement de la Loi contre les Pédophiles" -> "justice pénale et sécurité" existe 4 fois


17 % des arêtes désignent plusieurs cibles possibles. Aucune arête cassée en revanche :le problème est l'ambiguïté, pas l'incohérence.

## Rôle des nœuds

In [5]:
est_parent = {t["parent"] for t in declares}

seuls = [t for t in topics if not t["parent"] and t["name"] not in est_parent]
racines = [t for t in topics if not t["parent"] and t["name"] in est_parent]
milieu = [t for t in topics if t["parent"] and t["name"] in est_parent]
feuilles = [t for t in topics if t["parent"] and t["name"] not in est_parent]

pd.DataFrame([
    {"rôle": "isolé (ni parent ni enfant)", "n": len(seuls)},
    {"rôle": "racine (enfants, pas de parent)", "n": len(racines)},
    {"rôle": "milieu (parent + enfants)", "n": len(milieu)},
    {"rôle": "feuille (parent, pas d'enfants)", "n": len(feuilles)},
])

,rôle,n
0,isolé (ni parent ni enfant),1029
1,"racine (enfants, pas de parent)",220
2,milieu (parent + enfants),297
3,"feuille (parent, pas d'enfants)",3294


**1029 nœuds (21 %) ne sont reliés à rien.** Et il y a **220 racines**, pas 5 commele suggère le `level 3` : le champ `level` et la structure réelle divergent.

## Connexité

In [7]:
adj = defaultdict(set)
for t in topics:
    adj[t["name"]]
    if t["parent"] and t["parent"] in noms:
        adj[t["name"]].add(t["parent"])
        adj[t["parent"]].add(t["name"])

vus, composantes = set(), []
for depart in adj:
    if depart in vus:
        continue
    pile, comp = [depart], []
    while pile:
        n = pile.pop()
        if n in vus:
            continue
        vus.add(n)
        comp.append(n)
        pile.extend(adj[n] - vus)
    composantes.append(comp)

composantes.sort(key=len, reverse=True)
plus_grande = len(composantes[0])

print(f"composantes connexes : {len(composantes)}")
print(f"la plus grande : {plus_grande} nœuds ({round(100 * plus_grande / len(adj))}% du graphe)")
print(f"top 5 : {[len(c) for c in composantes[:5]]}")

composantes connexes : 1244
la plus grande : 394 nœuds (8% du graphe)
top 5 : [394, 243, 232, 153, 124]


**Ce n'est pas un graphe, ce sont 1244 graphes séparés.** Depuis n'importe quel nœud,on n'atteint au mieux que 8 % du corpus.

## Cycles

In [9]:
by_name = defaultdict(list)
for t in topics:
    by_name[t["name"]].append(t)


def boucle(t, vus=None):
    vus = vus or set()
    if t["name"] in vus:
        return True
    vus.add(t["name"])
    p = t["parent"]
    return boucle(by_name[p][0], vus) if p and p in by_name else False


n_cycles = sum(1 for t in topics if boucle(t))
print("topics dans un cycle :", n_cycles)
print("auto-références :", sum(1 for t in topics if t["parent"] == t["name"]))

topics dans un cycle : 78
auto-références : 1


Une requête récursive sans garde-fou boucle à l'infini sur ces 78 topics.

## Doublons de noms

In [10]:
dups = {n: c for n, c in noms.items() if c > 1}
print(f"{len(dups)} noms dupliqués, {sum(dups.values())} topics concernés\n")

pd.DataFrame([
    {"nom": n, "exemplaires": c, "arêtes entrantes": sum(1 for t in topics if t["parent"] == n)}
    for n, c in dups.items()
]).sort_values("arêtes entrantes", ascending=False).head(8)

18 noms dupliqués, 42 topics concernés



,nom,exemplaires,arêtes entrantes
0,gouvernance démocratique et institutions,3,92
1,politique environnementale et transition écolo...,2,75
15,justice fiscale et redistribution,2,54
14,politiques fiscales et redistribution,2,50
3,éducation et formation,3,48
11,fiscalité et impôts des ménages,3,43
6,santé publique et système de soins,2,43
12,transition écologique et environnement,2,43


Les doublons se concentrent sur les nœuds **les plus référencés** : l'ambiguïté frappelà où la structure compte le plus.

## Verdict

In [11]:
pd.DataFrame([
    ("nœuds", len(topics), "", ""),
    ("arêtes déclarées", len(declares), "", ""),
    ("arêtes ambiguës", len(ambigus), f"{round(100 * len(ambigus) / len(declares))}%", "BLOQUANT"),
    ("arêtes cassées", len(casses), "", "ok"),
    ("nœuds isolés", len(seuls), f"{round(100 * len(seuls) / len(topics))}%", "DÉGRADÉ"),
    ("racines réelles", len(racines), "vs 5 selon level 3", "DÉGRADÉ"),
    ("composantes connexes", len(composantes), f"la plus grande = {round(100 * plus_grande / len(adj))}%", "BLOQUANT"),
    ("topics en cycle", n_cycles, "", "BLOQUANT"),
    ("noms dupliqués", len(dups), f"{sum(dups.values())} topics", "BLOQUANT"),
    ("documents analysés", len(docs), f"sur {len(df)}", "INFO"),
], columns=["mesure", "valeur", "détail", "sévérité"])

,mesure,valeur,détail,sévérité
0,nœuds,4840,,
1,arêtes déclarées,3591,,
2,arêtes ambiguës,622,17%,BLOQUANT
3,arêtes cassées,0,,ok
4,nœuds isolés,1029,21%,DÉGRADÉ
5,racines réelles,220,vs 5 selon level 3,DÉGRADÉ
6,composantes connexes,1244,la plus grande = 8%,BLOQUANT
7,topics en cycle,78,,BLOQUANT
8,noms dupliqués,18,42 topics,BLOQUANT
9,documents analysés,100,sur 1524,INFO
